# Homework 2 — Part 1: Data Cleaning & Validation

**Goal:** turn the intentionally dirty `ohlc_hw` table into a trustworthy `ohlc_hw_cleaned` table.

Workflow used in this notebook:

1. **Load and inspect** the raw table (never modified).
2. **Normalize** ticker labels and dates *before* grouping or deduplicating.
3. **Diagnose**: run a set of detector functions and save a *Data Quality Report* **before** any repair.
4. **Clean**: remove duplicates, repair or quarantine bad values, and keep a repair flag for every change.
5. **Validate**: rerun every detector on the cleaned data, check hard rules, and save the result to the database.

The supplied `ohlc` table is used only as a **reference** (trading calendar, raw-price check and date-specific
adjustment factors). It never replaces the dirty dataset.

In [1]:
import re
import sqlite3
from contextlib import closing
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", None)   # show repair notes and evidence in full

# Local working copy of the course database (gitignored, never committed)
DB_PATH = Path("data/data.db")
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Download the bCourses database and place it at {DB_PATH.resolve()}")

# Column groups used throughout the notebook
KEY = ["ticker", "ts"]
PRICE_COLUMNS = ["open", "high", "low", "close"]          # raw prices
ALL_PRICE_COLUMNS = PRICE_COLUMNS + ["adj_close"]          # raw + adjusted prices
VALUE_COLUMNS = ALL_PRICE_COLUMNS + ["volume"]             # every measured value

## 1.1 Load and inspect the data

In [2]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    reference = pd.read_sql_query("SELECT * FROM ohlc", conn)

# Remember each row's position in the raw table so every later decision can be traced back
raw.insert(0, "row_id", np.arange(len(raw)))

print(f"ohlc_hw (dirty):   {len(raw):,} rows")
print(f"ohlc (reference):  {len(reference):,} rows")
raw.info()

ohlc_hw (dirty):   397,711 rows
ohlc (reference):  368,251 rows
<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   row_id     397711 non-null  int64  
 1   ts         397711 non-null  str    
 2   open       397711 non-null  float64
 3   high       397711 non-null  float64
 4   low        397711 non-null  float64
 5   close      397711 non-null  float64
 6   adj_close  377856 non-null  float64
 7   volume     397711 non-null  int64  
 8   ticker     397711 non-null  str    
dtypes: float64(5), int64(2), str(2)
memory usage: 34.7 MB


In [3]:
display(raw.head())
display(raw[VALUE_COLUMNS].describe().T)

,row_id,ts,open,high,low,close,adj_close,volume,ticker
0,0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,count,mean,std,min,25%,50%,75%,max
open,397711.0,4.628548e+01,6.713259e+01,0.000000,4.209938e+00,19.85,57.53,6.042600e+02
high,397711.0,4.674766e+01,6.777821e+01,0.005208,4.240000e+00,20.06,58.13,6.086300e+02
low,397711.0,4.581628e+01,6.647718e+01,0.004801,4.160000e+00,19.63,56.95,5.978800e+02
close,397711.0,4.629290e+01,6.714188e+01,0.005208,4.200000e+00,19.86,57.53,6.054000e+02
adj_close,377856.0,5.165225e+01,1.945840e+02,0.002945,1.390000e+00,11.96,43.77,1.044251e+04
volume,397711.0,3.097773e+07,1.266237e+08,0.000000,1.730700e+06,4648200.00,11023796.50,9.276606e+09


**Column meanings**

| column | meaning |
|---|---|
| `ts` | observation (trading) date, stored as text `YYYY-MM-DD` |
| `ticker` | stock symbol — **dirty**: some labels contain HTML markup |
| `open`, `high`, `low`, `close` | raw daily prices (in this data set they are already split-adjusted) |
| `adj_close` | vendor adjusted close (split **and** dividend adjusted) — used for returns in the reports |
| `volume` | number of shares traded that day |

In [4]:
print("Date range:", raw["ts"].min(), "to", raw["ts"].max())
print("Distinct raw ticker labels:", raw["ticker"].nunique())
print("Examples of raw labels:")
for label in raw["ticker"].drop_duplicates().head(3).tolist() + ["AAPL"]:
    print("   ", repr(label))

Date range: 1962-01-02 to 2024-12-06
Distinct raw ticker labels: 150
Examples of raw labels:
    'CAT'
    'CVX'
    'HD'
    'AAPL'


### Normalize ticker labels and dates first

The raw labels mix plain symbols with HTML such as `<a ...><span style="color: red">AAPL</span></a>`.
If we grouped or deduplicated *before* normalizing, the same stock would be split into 5 different "tickers"
and duplicates would be missed. We keep the original label in `original_ticker` and save an audit table.

In [5]:
def normalize_tickers(labels: pd.Series) -> pd.Series:
    '''Remove HTML tags and whitespace from ticker labels and make them upper case.'''
    without_tags = labels.str.replace(r"<[^>]+>", "", regex=True)
    return without_tags.str.strip().str.upper()


data = raw.copy()
data["original_ticker"] = data["ticker"]
data["ticker"] = normalize_tickers(data["ticker"])

# Audit: which raw label became which ticker, and how often
ticker_audit = (data.groupby(["original_ticker", "ticker"]).size()
                .reset_index(name="row_count").sort_values(["ticker", "original_ticker"]))
print("Rows whose label changed:", (data["original_ticker"] != data["ticker"]).sum())
display(ticker_audit.head(6))

tickers = sorted(data["ticker"].unique())
print(f"{len(tickers)} tickers after normalization:", tickers)
print("Same set as the reference table:", set(tickers) == set(reference["ticker"]))

Rows whose label changed: 39936


,original_ticker,ticker,row_count
0,"<a target=""_blank""><span style=""color: blue"">AAPL</span></a>",AAPL,307
30,"<a target=""_blank""><span style=""color: red"">AAPL</span></a>",AAPL,318
60,"<a target=""_blank""><span style=""font-weight: bold; color: blue"">AAPL</span></a>",AAPL,306
90,"<a target=""_blank""><span style=""font-weight: bold; color: red"">AAPL</span></a>",AAPL,264
120,AAPL,AAPL,10747
1,"<a target=""_blank""><span style=""color: blue"">AMGN</span></a>",AMGN,294


30 tickers after normalization: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


Same set as the reference table: True


In [6]:
# Dates: check that every ts is a real calendar date written as YYYY-MM-DD
parsed_dates = pd.to_datetime(data["ts"], format="%Y-%m-%d", errors="coerce")
print("Unparseable / non-ISO dates:", parsed_dates.isna().sum())

# Sort by ticker and date: required before any return calculation or filling
data = data.sort_values(KEY + ["row_id"]).reset_index(drop=True)

coverage = data.groupby("ticker")["ts"].agg(first_date="min", last_date="max", rows="size")
display(coverage)

Unparseable / non-ISO dates: 0


,first_date,last_date,rows
ticker,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


Note that the tickers' histories **end on different dates** (most on 2024-10-18, a few on 2024-12-06).
That is a property of the supplied data, not a fault, and it is disclosed again in the reports.

## 1.2 Detect data quality issues

Every detector returns rows in the same shape, so they can be stacked into one report:

`ticker | issue_type | issue_date | description | severity`

One observation can have several issues (for example a missing `adj_close` **and** zero volume).

In [7]:
REPORT_COLUMNS = ["ticker", "issue_type", "issue_date", "description", "severity"]


def make_issue_rows(rows: pd.DataFrame, issue_type: str, description, severity: str) -> pd.DataFrame:
    '''Convert flagged data rows into Data Quality Report rows.'''
    return pd.DataFrame({
        "ticker": rows["ticker"].to_numpy(),
        "issue_type": issue_type,
        "issue_date": rows["ts"].to_numpy(),
        "description": description,
        "severity": severity,
    }, columns=REPORT_COLUMNS)

### Formatting and duplicate records

In [8]:
def detect_ticker_markup(data: pd.DataFrame) -> pd.DataFrame:
    '''Rows whose raw ticker label needed normalization (HTML, whitespace or case).'''
    changed = data[data["original_ticker"] != data["ticker"]]
    return make_issue_rows(changed, "ticker_markup",
                           "Raw ticker label contained markup/whitespace; normalized", "low")


def detect_exact_duplicates(data: pd.DataFrame) -> pd.DataFrame:
    '''Extra copies of a row whose ticker, date and all values are identical to an earlier row.'''
    is_copy = data.duplicated(subset=KEY + VALUE_COLUMNS, keep="first")
    return make_issue_rows(data[is_copy], "exact_duplicate",
                           "Identical copy of another record (after ticker normalization)", "low")


def detect_conflicting_keys(data: pd.DataFrame) -> pd.DataFrame:
    '''Rows that share a (ticker, ts) key with a row holding DIFFERENT values.'''
    distinct_rows = data.drop_duplicates(subset=KEY + VALUE_COLUMNS)
    in_conflict = distinct_rows.duplicated(subset=KEY, keep=False)
    return make_issue_rows(distinct_rows[in_conflict], "conflicting_duplicate_key",
                           "Another record for the same ticker and date has different values", "high")


print("Exact duplicate rows:", len(detect_exact_duplicates(data)))
print("Rows in conflicting (ticker, ts) keys:", len(detect_conflicting_keys(data)))

Exact duplicate rows: 29460
Rows in conflicting (ticker, ts) keys: 0


### Missing data — missing cells

In [9]:
def detect_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    '''One issue row per missing cell in the price and volume columns.'''
    reports = []
    for column in VALUE_COLUMNS:
        rows = data[data[column].isna()]
        severity = "low" if column == "volume" else "medium"
        reports.append(make_issue_rows(rows, f"missing_{column}", f"{column} is missing", severity))
    return pd.concat(reports, ignore_index=True)


print("Missing cells per column:")
display(data[VALUE_COLUMNS].isna().sum().to_frame("missing_cells").T)

Missing cells per column:


,open,high,low,close,adj_close,volume
missing_cells,0,0,0,0,19855,0


### Missing data — missing trading days

Missing **cells** and missing **rows (dates)** are different checks. For the calendar we use the assignment's
allowed reference: the dates that the supplied `ohlc` table has **for each ticker** over that ticker's history.
(Business weekdays would wrongly count exchange holidays as gaps.) A date in the calendar with no row in our data
is a missing trading day. Detecting it does not require inventing a price for it.

In [10]:
def find_missing_trading_days(data: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    '''Return (ticker, ts) pairs that are in the reference calendar but absent from data.'''
    observed = data[KEY].drop_duplicates()
    expected = calendar[KEY].drop_duplicates()
    expected = expected[expected["ticker"].isin(observed["ticker"])]   # only tickers we actually have
    merged = expected.merge(observed, on=KEY, how="left", indicator=True)
    missing = merged.loc[merged["_merge"] == "left_only", KEY]
    return missing.sort_values(KEY).reset_index(drop=True)


def summarize_missing_trading_days(data: pd.DataFrame, calendar: pd.DataFrame) -> list[str]:
    '''Sentences "Ticker X is missing Y trading days between [start] and [end]" for every ticker.'''
    missing = find_missing_trading_days(data, calendar)
    sentences = []
    for ticker in sorted(data["ticker"].unique()):
        ticker_calendar = calendar.loc[calendar["ticker"] == ticker, "ts"]
        count = (missing["ticker"] == ticker).sum()
        sentences.append(f"Ticker {ticker} is missing {count} trading days between "
                         f"{ticker_calendar.min()} and {ticker_calendar.max()}")
    return sentences


def detect_missing_trading_days(data: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    '''One issue row per missing trading date (a separate row for each date).'''
    missing = find_missing_trading_days(data, calendar)
    return make_issue_rows(missing, "missing_trading_day",
                           "No record for a date that is in the ticker's reference calendar", "medium")


# Dates in our data that the calendar does NOT know would also be suspicious
unknown_dates = data[KEY].drop_duplicates().merge(reference[KEY], on=KEY, how="left", indicator=True)
print("Dates not in the reference calendar:", (unknown_dates["_merge"] == "left_only").sum())

for sentence in summarize_missing_trading_days(data, reference):
    print(sentence)

Dates not in the reference calendar: 0


Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

The real data has **zero** missing trading days. That is a legitimate result, so we do not invent gaps.
To prove the detector works, here it is on a tiny example with one deliberately removed date:

In [11]:
toy_calendar = pd.DataFrame({"ticker": "TOY",
                             "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"]})
toy_with_gap = toy_calendar.drop(index=2)          # remove 2024-01-04 on purpose
display(find_missing_trading_days(toy_with_gap, toy_calendar))
print(summarize_missing_trading_days(toy_with_gap, toy_calendar)[0])

,ticker,ts
0,TOY,2024-01-04


Ticker TOY is missing 1 trading days between 2024-01-02 and 2024-01-08


### Calendar sanity check — weekend dates

Comparing with the reference calendar cannot catch a problem that the reference shares. So we also check the dates
themselves: US exchanges do not trade on Saturdays or Sundays.

In [12]:
def detect_weekend_dates(data: pd.DataFrame) -> pd.DataFrame:
    '''Rows dated on a Saturday or Sunday, when US exchanges are closed.'''
    weekend = pd.to_datetime(data["ts"]).dt.day_of_week >= 5
    return make_issue_rows(data[weekend], "weekend_date",
                           "Dated on a weekend (exchange closed); likely the Friday session stamped on Sunday "
                           "(no Friday rows in these weeks). Kept; cannot verify.", "medium")


unique_keys = data.drop_duplicates(subset=KEY)
weekend_rows = unique_keys[pd.to_datetime(unique_keys["ts"]).dt.day_of_week >= 5]
print(f"Weekend-dated rows: {len(weekend_rows)} on {weekend_rows['ts'].nunique()} dates "
      f"({weekend_rows['ts'].min()} to {weekend_rows['ts'].max()}), {weekend_rows['ticker'].nunique()} tickers")
print("Day names:", pd.to_datetime(weekend_rows["ts"]).dt.day_name().value_counts().to_dict())
print("Also in the reference table:",
      int((pd.to_datetime(reference["ts"]).dt.day_of_week >= 5).sum()), "rows")

# Evidence: sessions per weekday in Jan-Apr 1974 for an affected ticker (KO) and an unaffected one (BA)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
for ticker in ["KO", "BA"]:
    in_period = unique_keys[(unique_keys["ticker"] == ticker) & unique_keys["ts"].between("1974-01-07", "1974-04-19")]
    sessions = pd.to_datetime(in_period["ts"]).dt.day_name().value_counts().reindex(weekday_order, fill_value=0)
    print(f"{ticker} sessions per weekday, 1974-01-07 to 1974-04-19:", sessions.to_dict())

Weekend-dated rows: 384 on 24 dates (1974-01-06 to 1975-04-20), 16 tickers
Day names: {'Sunday': 384}
Also in the reference table: 384 rows
KO sessions per weekday, 1974-01-07 to 1974-04-19: {'Monday': 15, 'Tuesday': 15, 'Wednesday': 15, 'Thursday': 14, 'Friday': 0, 'Saturday': 0, 'Sunday': 13}


BA sessions per weekday, 1974-01-07 to 1974-04-19: {'Monday': 15, 'Tuesday': 15, 'Wednesday': 15, 'Thursday': 14, 'Friday': 0, 'Saturday': 0, 'Sunday': 0}


**Finding:** 384 rows on 24 Sundays (Jan–Apr 1974 and Feb–Apr 1975) for 16 tickers; the reference table has
exactly the same rows. In those weeks **no ticker has a Friday row**: the 16 affected tickers (e.g. KO) have a
Sunday row instead, and the others (e.g. BA) simply have no session for that day. The Sunday prices are ordinary
trading bars that fit smoothly between Thursday and Monday, so the most likely explanation is that the vendor stamped the
**Friday session with the following Sunday's date**. The order of sessions (Thu → "Sun" → Mon) is still correct.

**Decision:** keep the rows and flag them (`weekend_date` column and `weekend_date` issue rows). Moving them to
Friday would need an external exchange calendar to confirm, and the row order (and therefore every return) is
unaffected. The missing Friday sessions of tickers like BA are consistent with the reference, so they are not
reported as gaps and no prices are invented for them. Both periods are decades outside every report window.
This is a documented uncertainty, not a repair.

### Outliers — extreme daily returns

Daily simple return on raw close, `(close - prev_close) / prev_close`, computed **within each ticker** after
sorting by date. Flag `abs(return - mean_return) > 3 * std_return` using the sample standard deviation
(`ddof=1`), both tails. Undefined returns (first day, missing/zero previous close) are excluded, and a ticker
with zero or undefined standard deviation gets no flags (stated explicitly). This is a **review flag, not a
deletion rule**.

In [13]:
def flag_return_outliers(data: pd.DataFrame, n_std: float = 3.0) -> pd.DataFrame:
    '''Add simple close-to-close returns and a 3-sigma review flag, computed within each ticker.'''
    result = data.sort_values(KEY).copy()
    returns = result.groupby("ticker")["close"].pct_change(fill_method=None)
    returns = returns.where(np.isfinite(returns))                 # inf (previous close 0) -> undefined
    result["simple_return"] = returns

    by_ticker = result.groupby("ticker")["simple_return"]
    result["mean_return"] = by_ticker.transform("mean")
    result["std_return"] = by_ticker.transform(lambda r: r.std(ddof=1))
    std_is_usable = result["std_return"].notna() & (result["std_return"] > 0)

    distance = (result["simple_return"] - result["mean_return"]).abs()
    result["return_outlier"] = (result["simple_return"].notna() & std_is_usable
                                & (distance > n_std * result["std_return"]))
    return result


def detect_return_outliers(data: pd.DataFrame) -> pd.DataFrame:
    '''Issue rows for returns more than 3 sample standard deviations from the ticker mean.'''
    flagged = flag_return_outliers(data)
    flagged = flagged[flagged["return_outlier"]]
    description = [f"Simple return {value:+.1%} is > 3 std from the ticker mean (review flag)"
                   for value in flagged["simple_return"]]
    return make_issue_rows(flagged, "return_outlier_3std", description, "medium")


# Explicit handling of zero / undefined standard deviation
outlier_check = flag_return_outliers(data.drop_duplicates(subset=KEY + VALUE_COLUMNS))
std_by_ticker = outlier_check.groupby("ticker")["std_return"].first()
print("Tickers with zero or undefined return std (no flags possible):",
      std_by_ticker[~(std_by_ticker > 0)].index.tolist())
print("3-sigma return flags:", outlier_check["return_outlier"].sum())

# Tiny example: constant prices have std 0, so no return can be flagged
constant = pd.DataFrame({"ticker": "FLAT", "ts": ["2024-01-02", "2024-01-03", "2024-01-04"],
                         "close": [10.0, 10.0, 10.0]})
display(flag_return_outliers(constant)[["ts", "close", "simple_return", "std_return", "return_outlier"]])

Tickers with zero or undefined return std (no flags possible): []
3-sigma return flags: 5056


,ts,close,simple_return,std_return,return_outlier
0,2024-01-02,10.0,NaN,0.0,False
1,2024-01-03,10.0,0.0,0.0,False
2,2024-01-04,10.0,0.0,0.0,False


### Outliers and inconsistencies — impossible prices, nonpositive values, zero volume

* **Price relationships:** a valid bar satisfies `low <= open <= high` and `low <= close <= high`.
  `high < close` or `low > close` is impossible.
* **Nonpositive / non-finite prices:** equity prices must be finite and > 0. This includes `adj_close`
  (but the raw OHLC bounds do **not** apply to `adj_close`, which lives on a different scale).
* **Zero volume** could mean stale data, so it is flagged for review.

In [14]:
def detect_price_relationship_issues(data: pd.DataFrame) -> pd.DataFrame:
    '''Rows that break low <= open <= high, low <= close <= high, or low <= high.'''
    checks = {
        "close_above_high": (data["close"] > data["high"], "close is above high (impossible)"),
        "close_below_low": (data["close"] < data["low"], "close is below low (impossible)"),
        "open_above_high": (data["open"] > data["high"], "open is above high (impossible)"),
        "open_below_low": (data["open"] < data["low"], "open is below low (impossible)"),
        "high_below_low": (data["high"] < data["low"], "high is below low (impossible)"),
    }
    reports = [make_issue_rows(data[mask], name, text, "high") for name, (mask, text) in checks.items()]
    return pd.concat(reports, ignore_index=True)


def detect_invalid_values(data: pd.DataFrame) -> pd.DataFrame:
    '''Nonpositive or non-finite prices (incl. adj_close), and negative or non-finite volume.'''
    reports = []
    for column in ALL_PRICE_COLUMNS:
        value = data[column]
        bad = value.notna() & (~np.isfinite(value) | (value <= 0))
        reports.append(make_issue_rows(data[bad], f"nonpositive_or_nonfinite_{column}",
                                       f"{column} is not a finite positive number", "high"))
    volume = data["volume"]
    bad_volume = volume.notna() & (~np.isfinite(volume) | (volume < 0))
    reports.append(make_issue_rows(data[bad_volume], "invalid_volume",
                                   "volume is negative or not finite", "high"))
    return pd.concat(reports, ignore_index=True)


def detect_zero_volume(data: pd.DataFrame) -> pd.DataFrame:
    '''Days with zero traded volume (possible stale or placeholder record).'''
    return make_issue_rows(data[data["volume"] == 0], "zero_volume",
                           "Zero volume: possibly stale record", "low")


unique_rows = data.drop_duplicates(subset=KEY + VALUE_COLUMNS)
print("Price relationship violations:")
display(detect_price_relationship_issues(unique_rows)["issue_type"].value_counts())
print("Invalid values:")
display(detect_invalid_values(unique_rows)["issue_type"].value_counts())
print("Zero-volume days:", len(detect_zero_volume(unique_rows)))

Price relationship violations:


issue_type
open_below_low    5
Name: count, dtype: int64

Invalid values:


issue_type
nonpositive_or_nonfinite_open    5
Name: count, dtype: int64

Zero-volume days: 75


In [15]:
# Show the rows behind the invalid values (all are open == 0 on a ticker's first day)
invalid_rows = detect_invalid_values(unique_rows)
display(unique_rows.merge(invalid_rows[["ticker", "issue_date"]].rename(columns={"issue_date": "ts"}), on=KEY)
        [KEY + VALUE_COLUMNS])
print("First date of each affected ticker:",
      coverage.loc[invalid_rows["ticker"], "first_date"].to_dict())

,ticker,ts,open,high,low,close,adj_close,volume
0,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840
1,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0
2,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800
3,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830
4,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000


First date of each affected ticker: {'CVX': '1962-01-02', 'JNJ': '1962-01-02', 'MCD': '1966-07-05', 'MRK': '1962-01-02', 'PG': '1962-01-02'}


### Inconsistent adjusted close

Missing `adj_close` is easy to see, but some `adj_close` values are also simply **wrong**. The adjustment factor
`adj_close / close` normally changes slowly (dividends) or occasionally (corporate actions). A factor that jumps far
away from its neighbours **for a single day** and comes back is a data error. We compare each day's factor with the
rolling median of the surrounding 21 days of the same ticker and flag deviations above 50%.

In [16]:
def find_inconsistent_adj_close(data: pd.DataFrame, tolerance: float = 0.5, window: int = 21) -> pd.Series:
    '''Boolean flag: adj_close / close deviates more than `tolerance` from its local rolling median.'''
    ordered = data.sort_values(KEY)
    factor = ordered["adj_close"] / ordered["close"]
    typical_factor = factor.groupby(ordered["ticker"]).transform(
        lambda f: f.rolling(window, center=True, min_periods=5).median())
    flag = factor.notna() & typical_factor.notna() & ((factor / typical_factor - 1).abs() > tolerance)
    return flag.reindex(data.index)


def detect_inconsistent_adj_close(data: pd.DataFrame) -> pd.DataFrame:
    '''Issue rows for adj_close values that break the ticker's local adjustment factor.'''
    flagged = data[find_inconsistent_adj_close(data)]
    ratio = flagged["adj_close"] / flagged["close"]
    description = [f"adj_close/close factor {value:.3f} far from neighbouring days" for value in ratio]
    return make_issue_rows(flagged, "adj_close_inconsistent", description, "high")


inconsistent = find_inconsistent_adj_close(unique_rows)
print("Inconsistent adj_close values:", inconsistent.sum())

# Evidence: compare with the reference table for the same (ticker, ts)
check = unique_rows[inconsistent].merge(reference[KEY + ["close", "adj_close"]], on=KEY, suffixes=("", "_ref"))
check["adj_close_ratio_to_reference"] = check["adj_close"] / check["adj_close_ref"]
print("Raw close equals reference close on these rows:", np.isclose(check["close"], check["close_ref"]).all())
display(check["adj_close_ratio_to_reference"].describe().to_frame().T)

Inconsistent adj_close values: 10418
Raw close equals reference close on these rows: True


,count,mean,std,min,25%,50%,75%,max
adj_close_ratio_to_reference,10418.0,14.080105,5.762877,1.0,9.185784,14.063553,19.059899,23.998439


The flagged `adj_close` values are 4–24 times the reference value while raw `close` matches the reference exactly:
the adjusted price itself is corrupted. Left in place, they would create fake +1,000% "returns" in the reports.

### The Data Quality Report (diagnosis **before** any repair)

**Severity choices**

| severity | issue types | why |
|---|---|---|
| **high** | impossible price relationships, nonpositive/non-finite values, corrupted `adj_close`, conflicting duplicate keys | the value is certainly wrong and would directly corrupt returns, volatility or drawdown |
| **medium** | missing prices / `adj_close`, missing trading days, weekend dates, 3σ return outliers | information is absent or needs human review; it may be legitimate (an outlier can be real news) |
| **low** | ticker markup, exact duplicates, zero volume, missing volume | easy to fix mechanically or does not affect price-based risk metrics |

In [17]:
def build_quality_report(data: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    '''Run every detector and stack the results into one Data Quality Report.'''
    detectors = [
        detect_ticker_markup(data),
        detect_exact_duplicates(data),
        detect_conflicting_keys(data),
        detect_missing_values(data),
        detect_missing_trading_days(data, calendar),
        detect_weekend_dates(data),
        detect_return_outliers(data.drop_duplicates(subset=KEY + VALUE_COLUMNS)),
        detect_price_relationship_issues(data),
        detect_invalid_values(data),
        detect_inconsistent_adj_close(data.drop_duplicates(subset=KEY + VALUE_COLUMNS)),
        detect_zero_volume(data),
    ]
    report = pd.concat(detectors, ignore_index=True)
    return report.sort_values(["ticker", "issue_date", "issue_type"]).reset_index(drop=True)


quality_report_before = build_quality_report(data, reference)
print(f"{len(quality_report_before):,} issue rows before cleaning")
display(quality_report_before.head(10))

issue_summary_before = (quality_report_before.groupby(["severity", "issue_type"]).size()
                        .rename("issue_rows").reset_index().sort_values(["severity", "issue_rows"]))
display(issue_summary_before)

# One observation can carry several issues: show how often that happens and one example
issues_per_observation = quality_report_before.groupby(["ticker", "issue_date"])["issue_type"].nunique()
print("Observations with more than one issue type:", (issues_per_observation > 1).sum())
example_key = issues_per_observation[issues_per_observation > 1].sort_values(ascending=False).index[0]
display(quality_report_before[(quality_report_before["ticker"] == example_key[0])
                              & (quality_report_before["issue_date"] == example_key[1])])

105,229 issue rows before cleaning


,ticker,issue_type,issue_date,description,severity
0,AAPL,ticker_markup,1980-12-19,Raw ticker label contained markup/whitespace; normalized,low
1,AAPL,return_outlier_3std,1980-12-26,Simple return +9.2% is > 3 std from the ticker mean (review flag),medium
2,AAPL,ticker_markup,1980-12-29,Raw ticker label contained markup/whitespace; normalized,low
3,AAPL,exact_duplicate,1981-01-02,Identical copy of another record (after ticker normalization),low
4,AAPL,exact_duplicate,1981-01-05,Identical copy of another record (after ticker normalization),low
5,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
6,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
7,AAPL,ticker_markup,1981-02-02,Raw ticker label contained markup/whitespace; normalized,low
8,AAPL,adj_close_inconsistent,1981-02-03,adj_close/close factor 10.254 far from neighbouring days,high
9,AAPL,ticker_markup,1981-02-04,Raw ticker label contained markup/whitespace; normalized,low


,severity,issue_type,issue_rows
1,high,nonpositive_or_nonfinite_open,5
2,high,open_below_low,5
0,high,adj_close_inconsistent,10418
5,low,zero_volume,81
3,low,exact_duplicate,29460
4,low,ticker_markup,39936
8,medium,weekend_date,413
7,medium,return_outlier_3std,5056
6,medium,missing_adj_close,19855


Observations with more than one issue type: 8800


,ticker,issue_type,issue_date,description,severity
909,AAPL,adj_close_inconsistent,1993-10-21,adj_close/close factor 15.776 far from neighbouring days,high
910,AAPL,exact_duplicate,1993-10-21,Identical copy of another record (after ticker normalization),low
911,AAPL,return_outlier_3std,1993-10-21,Simple return +9.0% is > 3 std from the ticker mean (review flag),medium
912,AAPL,ticker_markup,1993-10-21,Raw ticker label contained markup/whitespace; normalized,low
913,AAPL,ticker_markup,1993-10-21,Raw ticker label contained markup/whitespace; normalized,low


### Injected examples for cases that do not occur in the real data

Several required checks find **zero** issues in `ohlc_hw` (missing raw prices, missing volume, `high < close`,
`low > close`, conflicting keys, missing dates, infinite values). A check that finds nothing is fine, but we must
show it *would* catch the problem. Below is a small fake ticker with each fault injected on purpose.
Its "reference" is the clean version of the same data (adjustment factor 0.5).

In [18]:
def make_toy_data():
    '''A small clean ticker (reference) and a copy with deliberately injected faults.'''
    dates = pd.bdate_range("2024-01-01", periods=10).strftime("%Y-%m-%d")
    close = np.array([100, 101, 102, 101, 103, 104, 103, 105, 106, 107], dtype=float)
    clean = pd.DataFrame({"ts": dates, "ticker": "TOY", "open": close - 0.5, "high": close + 1.0,
                          "low": close - 1.0, "close": close, "adj_close": close * 0.5, "volume": 1000})
    dirty = clean.copy()
    dirty.loc[0, "open"] = np.nan                     # leading gap: nothing earlier to forward-fill
    dirty.loc[2, "close"] = np.nan                    # missing raw price
    dirty.loc[3, ["open", "high", "low", "close"]] = np.nan   # ALL raw prices missing -> drop
    dirty.loc[4, "close"] = 110.0                     # high < close (impossible)
    dirty.loc[5, "low"] = 105.0                       # low > close (impossible)
    dirty.loc[6, "volume"] = np.nan                   # missing volume
    dirty.loc[7, "adj_close"] = np.nan                # missing adj_close
    dirty.loc[8, "high"] = np.inf                     # non-finite price
    dirty = dirty.drop(index=9)                       # missing trading day (last date removed)
    conflict = dirty.loc[[1]].assign(close=150.0, high=151.0)   # conflicting copy of 2024-01-02's key
    exact_copy = dirty.loc[[6]]                                  # exact duplicate
    dirty = pd.concat([dirty, conflict, exact_copy], ignore_index=True)
    dirty.insert(0, "row_id", np.arange(len(dirty)))
    dirty["original_ticker"] = " toy "                # label needing normalization
    return clean, dirty


toy_reference, toy_dirty = make_toy_data()
display(toy_dirty)
toy_report = build_quality_report(toy_dirty, toy_reference)
display(toy_report)

,row_id,ts,ticker,open,high,low,close,adj_close,volume,original_ticker
0,0,2024-01-01,TOY,NaN,101.0,99.0,100.0,50.0,1000.0,toy
1,1,2024-01-02,TOY,100.5,102.0,100.0,101.0,50.5,1000.0,toy
2,2,2024-01-03,TOY,101.5,103.0,101.0,NaN,51.0,1000.0,toy
3,3,2024-01-04,TOY,NaN,NaN,NaN,NaN,50.5,1000.0,toy
4,4,2024-01-05,TOY,102.5,104.0,102.0,110.0,51.5,1000.0,toy
5,5,2024-01-08,TOY,103.5,105.0,105.0,104.0,52.0,1000.0,toy
6,6,2024-01-09,TOY,102.5,104.0,102.0,103.0,51.5,NaN,toy
7,7,2024-01-10,TOY,104.5,106.0,104.0,105.0,NaN,1000.0,toy
8,8,2024-01-11,TOY,105.5,inf,105.0,106.0,53.0,1000.0,toy
9,9,2024-01-02,TOY,100.5,151.0,100.0,150.0,50.5,1000.0,toy


,ticker,issue_type,issue_date,description,severity
0,TOY,missing_open,2024-01-01,open is missing,medium
1,TOY,ticker_markup,2024-01-01,Raw ticker label contained markup/whitespace; normalized,low
2,TOY,conflicting_duplicate_key,2024-01-02,Another record for the same ticker and date has different values,high
3,TOY,conflicting_duplicate_key,2024-01-02,Another record for the same ticker and date has different values,high
4,TOY,ticker_markup,2024-01-02,Raw ticker label contained markup/whitespace; normalized,low
5,TOY,ticker_markup,2024-01-02,Raw ticker label contained markup/whitespace; normalized,low
6,TOY,missing_close,2024-01-03,close is missing,medium
7,TOY,ticker_markup,2024-01-03,Raw ticker label contained markup/whitespace; normalized,low
8,TOY,missing_close,2024-01-04,close is missing,medium
9,TOY,missing_high,2024-01-04,high is missing,medium


## 1.3 Clean the data

Repairs are applied in this order, always on data **sorted by ticker and date** and **within each ticker**
(we never borrow a value from another ticker):

| step | rule | decision |
|---|---|---|
| 1 | exact duplicates | keep one copy |
| 2 | conflicting `(ticker, ts)` keys | keep the version whose raw OHLC and volume match the `ohlc` reference; if no unique match, quarantine the key |
| 3 | all four raw prices missing | drop the row **before** any filling |
| 4 | impossible / nonpositive / non-finite raw price | **Option 1**: set the offending value to NaN, then forward-fill (step 5) |
| 5 | missing raw prices | forward-fill within the ticker, then keep the filled value inside the day's high–low range; leading gaps are quarantined |
| 6 | missing volume | fill 0 and set `volume_filled` (unknown volume is **not** evidence of no trading) |
| 7 | confirmed price spike errors | interpolate between the valid neighbouring days |
| 8 | missing or corrupted `adj_close` | rebuild as `close × factor`, where factor = `adj_close / close` of the **same** `(ticker, ts)` in `ohlc`; quarantine if no trusted factor |

**Why Option 1 for impossible prices?** The close is the value we cannot trust when `close > high`; carrying the
last known close is the most conservative choice and is the same rule we use for any missing price. We then clip it
into the day's observed `[low, high]` range, because forward filling alone can violate today's range.
Option 2 is equivalent for the close; Option 3 (rebuild from O/H/L) would invent a value we cannot verify.

**Why rebuild `adj_close` from a factor instead of forward-filling?** The factor changes on ex-dividend and
corporate-action dates, so carrying yesterday's adjusted price (or factor) across such a date would be wrong.
The reference table gives the vendor's factor for that exact date. Because our raw close matches the reference
close exactly, `close × factor` keeps one consistent adjustment convention.

In [19]:
def add_note(data: pd.DataFrame, mask: pd.Series, note: str) -> None:
    '''Append a repair note to the rows selected by mask (in place).'''
    old = data.loc[mask, "repair_note"]
    data.loc[mask, "repair_note"] = np.where(old == "", note, old + "; " + note)


def quarantine_rows(rows: pd.DataFrame, reason: str) -> pd.DataFrame:
    '''Copy rows into quarantine format together with the reason.'''
    return rows.assign(quarantine_reason=reason)


def remove_exact_duplicates(data: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    '''Step 1: keep the first copy of identical records.'''
    is_copy = data.duplicated(subset=KEY + VALUE_COLUMNS, keep="first")
    return data[~is_copy].copy(), int(is_copy.sum())


def resolve_conflicting_keys(data: pd.DataFrame, reference: pd.DataFrame):
    '''Step 2: per conflicting key keep the single row that matches the reference, else quarantine the key.'''
    in_conflict = data.duplicated(subset=KEY, keep=False)
    if not in_conflict.any():
        return data, data.iloc[0:0].assign(quarantine_reason=""), 0

    candidates = data[in_conflict].merge(reference[KEY + PRICE_COLUMNS + ["volume"]],
                                         on=KEY, how="left", suffixes=("", "_ref"))
    candidates["matches_reference"] = np.isclose(candidates[PRICE_COLUMNS].to_numpy(dtype=float),
                                                 candidates[[c + "_ref" for c in PRICE_COLUMNS]].to_numpy(dtype=float)
                                                 ).all(axis=1) & (candidates["volume"] == candidates["volume_ref"])
    matches_per_key = candidates.groupby(KEY)["matches_reference"].transform("sum")
    keep_ids = candidates.loc[candidates["matches_reference"] & (matches_per_key == 1), "row_id"]
    unresolved_ids = candidates.loc[matches_per_key != 1, "row_id"]

    removed_count = int(in_conflict.sum() - len(keep_ids) - len(unresolved_ids))
    quarantined = quarantine_rows(data[data["row_id"].isin(unresolved_ids)],
                                  "conflicting (ticker, ts) values; no unique match in reference")
    kept = data[~in_conflict | data["row_id"].isin(keep_ids)].copy()
    add_note(kept, kept["row_id"].isin(keep_ids), "conflicting key resolved using ohlc reference")
    return kept, quarantined, removed_count


def drop_rows_missing_all_prices(data: pd.DataFrame):
    '''Step 3: rows with no raw price at all carry no usable information.'''
    all_missing = data[PRICE_COLUMNS].isna().all(axis=1)
    return data[~all_missing].copy(), data[all_missing].copy()


def null_invalid_prices(data: pd.DataFrame) -> pd.DataFrame:
    '''Step 4 (Option 1): replace impossible, nonpositive or non-finite raw prices with NaN.'''
    data = data.copy()
    for column in PRICE_COLUMNS:
        value = data[column]
        bad = value.notna() & (~np.isfinite(value) | (value <= 0))
        add_note(data, bad, f"{column} was not finite/positive; forward-filled")
        data.loc[bad, column] = np.nan
    for column in ["open", "close"]:
        outside = (data[column] > data["high"]) | (data[column] < data["low"])
        add_note(data, outside, f"{column} outside high-low range; forward-filled")
        data.loc[outside, column] = np.nan
    high_below_low = data["high"] < data["low"]
    add_note(data, high_below_low, "high below low; both forward-filled")
    data.loc[high_below_low, ["high", "low"]] = np.nan
    return data


def fill_missing_prices(data: pd.DataFrame):
    '''Step 5: forward-fill raw prices within each ticker and keep each bar internally valid.'''
    data = data.sort_values(KEY).copy()
    was_missing = data[PRICE_COLUMNS].isna()
    data[PRICE_COLUMNS] = data.groupby("ticker")[PRICE_COLUMNS].ffill()

    # A leading gap has no earlier price in the same ticker: quarantine instead of guessing
    leading_gap = data[PRICE_COLUMNS].isna().any(axis=1)
    quarantined = quarantine_rows(data[leading_gap], "leading gap: no earlier price in this ticker to forward-fill")
    data = data[~leading_gap].copy()
    was_missing = was_missing.loc[data.index]

    # Forward-filled open/close must stay inside today's observed range
    for column in ["open", "close"]:
        filled = was_missing[column]
        data.loc[filled, column] = data.loc[filled, column].clip(data.loc[filled, "low"], data.loc[filled, "high"])
    # If high or low itself was filled, widen it so the bar contains open and close
    range_filled = was_missing["high"] | was_missing["low"]
    data.loc[range_filled, "high"] = data.loc[range_filled, PRICE_COLUMNS].max(axis=1)
    data.loc[range_filled, "low"] = data.loc[range_filled, PRICE_COLUMNS].min(axis=1)

    data["price_filled"] = data["price_filled"] | was_missing.any(axis=1)
    data["close_repaired"] = was_missing["close"]      # adj_close must then be rebuilt as well (step 8)
    add_note(data, was_missing.any(axis=1) & ~data["repair_note"].str.contains("forward-filled"),
             "missing raw price forward-filled")
    return data, quarantined


def fill_missing_volume(data: pd.DataFrame) -> pd.DataFrame:
    '''Step 6: unknown volume -> 0 with a flag (0 here means "unknown", not "no trading").'''
    data = data.copy()
    missing = data["volume"].isna()
    data["volume_filled"] = missing
    data.loc[missing, "volume"] = 0
    add_note(data, missing, "volume unknown; filled with 0")
    return data

### Investigating the extreme returns (step 7)

Before repairing anything we look for **evidence** about each 3σ flag:

* **Confirmed print error:** the close moves by 50% or more in one day **and** the next day's close is back within
  5% of the previous close, i.e. a one-day spike that fully reverses. Real crashes (1987, 2008, March 2020)
  move less and do not snap back to the exact previous level.
* **Possible corporate action (split, merger/spin-off, special dividend):** the adjustment factor
  `adj_close / close` changes by more than 5% on the same day, so the raw and adjusted moves differ. Note that this
  data set's raw close is already split-adjusted, so splits themselves rarely show up as raw-price jumps. We have no
  corporate-actions or news feed, so we cannot tell a merger from a split or a special dividend: we report
  "possible" and do not name an event. If the factor is back to its previous value the next day, it is instead a
  **one-day blip in the vendor's factor** (e.g. AAPL 1990-10-19, where `adj_close` equals `close` for one day).
* **Known limitation:** the reference factor has a few such small one-day blips before 1991 (the largest ~26%,
  below the 50% corruption threshold). They come from the vendor reference itself, so we cannot tell which value is
  right; we keep them and the note records it. None falls inside a report window.
* **Market-wide move:** at least 5 of the 30 tickers are flagged on the same date (e.g. 1987-10-19, 2020-03-16).
* **Otherwise:** a single-stock move whose raw close agrees with the reference table. We keep it and say that the
  cause (news, earnings, …) is **not verified**.

In [20]:
outlier_view = flag_return_outliers(data.drop_duplicates(subset=KEY + VALUE_COLUMNS))
group = outlier_view.groupby("ticker")
outlier_view["prev_close"] = group["close"].shift(1)
outlier_view["next_close"] = group["close"].shift(-1)
outlier_view["next_reverts"] = (outlier_view["next_close"] / outlier_view["prev_close"] - 1).abs() < 0.05

print("Largest absolute 3-sigma moves:")
largest = outlier_view[outlier_view["return_outlier"]].copy()
largest["abs_return"] = largest["simple_return"].abs()
largest = largest.sort_values("abs_return", ascending=False)
display(largest[KEY + ["prev_close", "close", "next_close", "simple_return", "next_reverts", "volume"]].head(8))

Largest absolute 3-sigma moves:


,ticker,ts,prev_close,close,next_close,simple_return,next_reverts,volume
343765,TRV,1976-10-28,4.61000,9.22000,4.69000,1.000000,True,0
345085,TRV,1981-08-10,6.03000,12.06000,6.03000,1.000000,True,0
344353,TRV,1978-12-20,4.02000,8.03000,4.09000,0.997512,True,0
230852,MCD,1968-01-02,0.70988,0.17490,0.17567,-0.753620,False,6755143
5419,AAPL,2000-09-29,0.95536,0.45982,0.43304,-0.518695,False,3128875900
345086,TRV,1981-08-11,12.06000,6.03000,6.13000,-0.500000,False,24800
343766,TRV,1976-10-29,9.22000,4.69000,4.66000,-0.491323,False,79200
344354,TRV,1978-12-21,8.03000,4.09000,4.09000,-0.490660,False,105600


The three TRV rows on top are textbook print errors: open, high, low **and** close are exactly double the
neighbouring days, volume is 0, and the price snaps back the next day. These are the only **confirmed** errors,
and we interpolate them. All other large moves (including the crash and rebound days) stay in the data
with a review flag.

In [21]:
def find_spike_errors(data: pd.DataFrame, min_jump: float = 0.5, revert_tolerance: float = 0.05) -> pd.Series:
    '''Confirmed one-day print errors: |return| >= min_jump and the next close returns to the previous close.'''
    ordered = data.sort_values(KEY)
    group = ordered.groupby("ticker")["close"]
    prev_close, next_close = group.shift(1), group.shift(-1)
    jump = ordered["close"] / prev_close - 1
    reverts = (next_close / prev_close - 1).abs() < revert_tolerance
    return ((jump.abs() >= min_jump) & reverts).reindex(data.index)


def interpolate_between_neighbours(data: pd.DataFrame, rows: pd.Series, columns: list[str]) -> None:
    '''Replace values on the selected rows by the midpoint of the previous and next day of the same ticker.'''
    group = data.groupby("ticker")
    for column in columns:
        midpoint = (group[column].shift(1) + group[column].shift(-1)) / 2
        data.loc[rows, column] = midpoint[rows]


def interpolate_spike_errors(data: pd.DataFrame) -> pd.DataFrame:
    '''Step 7: replace the raw OHLC of confirmed spike days by the midpoint of the neighbouring days.

    adj_close of these rows is interpolated at the end of step 8, after the neighbours' own
    adj_close values have been repaired (otherwise a corrupted neighbour would leak in).
    '''
    data = data.sort_values(KEY).copy()
    spike = find_spike_errors(data)
    interpolate_between_neighbours(data, spike, PRICE_COLUMNS)
    data["price_interpolated"] = spike
    add_note(data, spike, "confirmed one-day print error (price doubled and reverted); "
                          "OHLC and adj_close interpolated from neighbouring days")
    return data

In [22]:
def repair_adj_close(data: pd.DataFrame, reference: pd.DataFrame):
    '''Step 8: rebuild missing or corrupted adj_close as close x reference factor of the same date.'''
    data = data.copy()
    adj = data["adj_close"]
    missing = adj.isna()
    invalid = adj.notna() & (~np.isfinite(adj) | (adj <= 0))
    inconsistent = find_inconsistent_adj_close(data)
    close_changed = data["close_repaired"]               # keep adj consistent with a repaired close
    # Interpolated spike days are handled separately below, so they are excluded here
    needs_repair = (missing | invalid | inconsistent | close_changed) & ~data["price_interpolated"]

    # Trusted factor: from the reference row with the same (ticker, ts) AND the same raw close.
    # If we repaired the close ourselves, the factor of that date is still the vendor's factor for that date.
    factors = reference[KEY + ["close", "adj_close"]].rename(columns={"close": "ref_close", "adj_close": "ref_adj"})
    merged = data[KEY + ["close"]].merge(factors, on=KEY, how="left")
    merged.index = data.index
    factor = merged["ref_adj"] / merged["ref_close"]
    same_close = np.isclose(merged["close"], merged["ref_close"], rtol=1e-6)
    trusted = (same_close | data["close_repaired"]) & np.isfinite(factor) & (factor > 0)

    repairable = needs_repair & trusted
    data.loc[repairable, "adj_close"] = data.loc[repairable, "close"] * factor[repairable]
    data["adj_close_repair"] = np.select(
        [missing & repairable, (invalid | inconsistent) & repairable, close_changed & repairable],
        ["was_missing", "was_inconsistent", "close_was_repaired"], default="")
    add_note(data, repairable, "adj_close = close x factor from ohlc (same ticker and date)")

    # Confirmed spike days: the reference carries the same bad print, so interpolate from repaired neighbours
    interpolate_between_neighbours(data, data["price_interpolated"], ["adj_close"])
    data.loc[data["price_interpolated"], "adj_close_repair"] = "interpolated"

    unresolved = needs_repair & ~trusted
    quarantined = quarantine_rows(data[unresolved], "adj_close missing/corrupted and no trusted reference factor")
    return data[~unresolved].copy(), quarantined

### Review flags kept with the cleaned data

In [23]:
def describe_return_outliers(data: pd.DataFrame, market_wide_count: int = 5) -> pd.DataFrame:
    '''Add return_outlier + outlier_note (evidence or stated uncertainty) computed on the repaired prices.'''
    result = flag_return_outliers(data)
    factor = result["adj_close"] / result["close"]
    factor_by_ticker = factor.groupby(result["ticker"])
    previous_factor, next_factor = factor_by_ticker.shift(1), factor_by_ticker.shift(-1)
    factor_change = (factor / previous_factor - 1).abs()
    # A factor that jumps and is back the next day is a vendor blip, not a corporate action
    factor_blip = (factor_change > 0.05) & ((next_factor / previous_factor - 1).abs() < 0.02)
    tickers_flagged_that_day = result.groupby("ts")["return_outlier"].transform("sum")

    is_outlier = result["return_outlier"]
    notes = np.select(
        [is_outlier & (tickers_flagged_that_day >= market_wide_count),
         is_outlier & factor_blip,
         is_outlier & (factor_change > 0.05),
         is_outlier],
        ["kept: market-wide move (" + tickers_flagged_that_day.astype(int).astype(str) + " tickers flagged that day)",
         "kept: one-day blip in the vendor adjustment factor (reverts next day); raw close agrees with ohlc",
         "kept: adjustment factor changed " + factor_change.map("{:.0%}".format)
         + " the same day and stays; possible corporate action (split, merger/spin-off, special dividend; "
           "not verified)",
         "kept: single-stock move; raw close agrees with ohlc reference; cause not verified"],
        default="")
    result["outlier_note"] = notes
    return result.drop(columns=["simple_return", "mean_return", "std_return"])

### Run the full cleaning pipeline

In [24]:
def recheck(data: pd.DataFrame, step: str) -> dict:
    '''Count the rule violations still present after a cleaning step.

    Raw bounds are checked on rows whose four raw prices are all present (values set to NaN
    on purpose are counted as "raw prices pending fill" until the forward-fill step).
    '''
    prices = data[PRICE_COLUMNS]
    complete = data[prices.notna().all(axis=1)]
    lowest_body = complete[["open", "close"]].min(axis=1)
    highest_body = complete[["open", "close"]].max(axis=1)
    adj = data["adj_close"]
    return {
        "after step": step,
        "rows": len(data),
        "raw prices pending fill": int(prices.isna().any(axis=1).sum()),
        "raw bound violations": int(((complete["low"] > lowest_body) | (complete["high"] < highest_body)
                                     | (complete["high"] < complete["low"])).sum()),
        "nonpositive/non-finite raw": int((~np.isfinite(complete[PRICE_COLUMNS]) | (complete[PRICE_COLUMNS] <= 0))
                                          .any(axis=1).sum()),
        "adj_close missing/invalid": int((adj.isna() | ~np.isfinite(adj) | (adj <= 0)).sum()),
        "adj_close inconsistent": int(find_inconsistent_adj_close(data).sum()),
    }


def clean_ohlc(data: pd.DataFrame, reference: pd.DataFrame):
    '''Run steps 1-8 and the review flags, rechecking the rules after every step.

    Returns (cleaned, quarantine, accounting, step_checks).
    '''
    work = data.copy()
    work["repair_note"] = ""
    work["price_filled"] = False
    accounting = {"raw_rows": len(work)}
    step_checks = [recheck(work, "0. start (normalized)")]

    work, accounting["exact_duplicates_removed"] = remove_exact_duplicates(work)
    step_checks.append(recheck(work, "1. remove exact duplicates"))
    work, quarantine_conflicts, accounting["conflicting_rows_removed"] = resolve_conflicting_keys(work, reference)
    step_checks.append(recheck(work, "2. resolve conflicting keys"))
    work, all_missing = drop_rows_missing_all_prices(work)
    accounting["all_prices_missing_removed"] = len(all_missing)
    step_checks.append(recheck(work, "3. drop rows missing all prices"))
    work = null_invalid_prices(work)
    step_checks.append(recheck(work, "4. invalid raw prices -> NaN"))
    work, quarantine_leading = fill_missing_prices(work)
    step_checks.append(recheck(work, "5. forward-fill raw prices"))
    work = fill_missing_volume(work)
    step_checks.append(recheck(work, "6. fill missing volume"))
    work = interpolate_spike_errors(work)
    step_checks.append(recheck(work, "7. interpolate confirmed spikes"))
    work, quarantine_adj = repair_adj_close(work, reference)
    step_checks.append(recheck(work, "8. repair adj_close"))
    work = describe_return_outliers(work)
    work["zero_volume"] = work["volume"] == 0
    work["weekend_date"] = pd.to_datetime(work["ts"]).dt.day_of_week >= 5
    work["volume"] = work["volume"].astype("int64")

    quarantine = pd.concat([quarantine_conflicts, quarantine_leading, quarantine_adj], ignore_index=True)
    accounting["quarantined"] = len(quarantine)
    accounting["retained"] = len(work)
    removed_total = (accounting["exact_duplicates_removed"] + accounting["conflicting_rows_removed"]
                     + accounting["all_prices_missing_removed"])
    assert accounting["retained"] + accounting["quarantined"] + removed_total == accounting["raw_rows"], accounting
    return work.sort_values(KEY).reset_index(drop=True), quarantine, accounting, pd.DataFrame(step_checks)

First on the injected toy example, so every repair rule can be checked by eye:

In [25]:
toy_clean, toy_quarantine, toy_accounting, toy_checks = clean_ohlc(toy_dirty, toy_reference)
print(toy_accounting)
print("Rules rechecked after every repair step:")
display(toy_checks)
display(toy_clean[["ts", "ticker"] + VALUE_COLUMNS + ["price_filled", "volume_filled", "adj_close_repair", "repair_note"]])
display(toy_quarantine[["ts", "ticker"] + VALUE_COLUMNS + ["quarantine_reason"]])

{'raw_rows': 11, 'exact_duplicates_removed': 1, 'conflicting_rows_removed': 1, 'all_prices_missing_removed': 1, 'quarantined': 1, 'retained': 7}
Rules rechecked after every repair step:


,after step,rows,raw prices pending fill,raw bound violations,nonpositive/non-finite raw,adj_close missing/invalid,adj_close inconsistent
0,0. start (normalized),11,3,2,1,1,0
1,1. remove exact duplicates,10,3,2,1,1,0
2,2. resolve conflicting keys,9,3,2,1,1,0
3,3. drop rows missing all prices,8,2,2,1,1,0
4,4. invalid raw prices -> NaN,8,5,0,0,1,0
5,5. forward-fill raw prices,7,0,0,0,1,0
6,6. fill missing volume,7,0,0,0,1,0
7,7. interpolate confirmed spikes,7,0,0,0,1,0
8,8. repair adj_close,7,0,0,0,0,0


,ts,ticker,open,high,low,close,adj_close,volume,price_filled,volume_filled,adj_close_repair,repair_note
0,2024-01-02,TOY,100.5,102.0,100.0,101.0,50.5,1000,False,False,,conflicting key resolved using ohlc reference
1,2024-01-03,TOY,101.5,103.0,101.0,101.0,50.5,1000,True,False,close_was_repaired,missing raw price forward-filled; adj_close = close x factor from ohlc (same ticker and date)
2,2024-01-05,TOY,102.5,104.0,102.0,102.0,51.0,1000,True,False,close_was_repaired,close outside high-low range; forward-filled; adj_close = close x factor from ohlc (same ticker and date)
3,2024-01-08,TOY,105.0,105.0,105.0,105.0,52.5,1000,True,False,close_was_repaired,open outside high-low range; forward-filled; close outside high-low range; forward-filled; adj_close = close x factor from ohlc (same ticker and date)
4,2024-01-09,TOY,102.5,104.0,102.0,103.0,51.5,0,False,True,,volume unknown; filled with 0
5,2024-01-10,TOY,104.5,106.0,104.0,105.0,52.5,1000,False,False,was_missing,adj_close = close x factor from ohlc (same ticker and date)
6,2024-01-11,TOY,105.5,106.0,105.0,106.0,53.0,1000,True,False,,high was not finite/positive; forward-filled


,ts,ticker,open,high,low,close,adj_close,volume,quarantine_reason
0,2024-01-01,TOY,NaN,101.0,99.0,100.0,50.0,1000.0,leading gap: no earlier price in this ticker to forward-fill


What happened in the toy example:

* 2024-01-01: leading gap in `open` → **quarantined** (nothing earlier to carry forward).
* 2024-01-02: the conflicting copy (close 150) does not match the reference → dropped; the matching row is kept.
* 2024-01-03: missing close → forward-filled from 01-02 and kept inside the day's range.
* 2024-01-04: all raw prices missing → **dropped** before filling. (It would otherwise be a fully invented bar.)
* 2024-01-05: close 110 > high 104 → set to NaN, forward-filled, clipped into `[low, high]`.
* 2024-01-08: low 105 > close 104 → close forward-filled and clipped up to the low (105).
* 2024-01-09: missing volume → 0 with `volume_filled = True`; the exact duplicate is removed.
* 2024-01-10: missing `adj_close` → rebuilt as `close × 0.5` from the reference factor.
* Every row whose close was repaired also gets `adj_close` rebuilt (`adj_close_repair = close_was_repaired`),
  so the adjusted price stays on the same convention as the repaired raw close.
* 2024-01-11: infinite high → NaN → forward-filled, range widened to contain open and close.
* 2024-01-12 was removed from the input; the detector reported it as a missing trading day and we did **not**
  invent a price for it.

The spike interpolation needs a one-day reversal, so here is a second tiny example:

In [26]:
spike_example = pd.DataFrame({
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04"], "ticker": "SPK",
    "open": [10.0, 20.0, 10.2], "high": [10.0, 20.0, 10.2], "low": [10.0, 20.0, 10.2],
    "close": [10.0, 20.0, 10.2], "adj_close": [5.0, 10.0, 5.1], "volume": [100, 0, 100]})
spike_fixed = interpolate_spike_errors(spike_example.assign(repair_note=""))
interpolate_between_neighbours(spike_fixed, spike_fixed["price_interpolated"], ["adj_close"])  # done in step 8
display(spike_fixed[["ts"] + ALL_PRICE_COLUMNS + ["price_interpolated"]])

,ts,open,high,low,close,adj_close,price_interpolated
0,2024-01-02,10.0,10.0,10.0,10.0,5.00,False
1,2024-01-03,10.1,10.1,10.1,10.1,5.05,True
2,2024-01-04,10.2,10.2,10.2,10.2,5.10,False


Now the real data:

In [27]:
cleaned, quarantine, accounting, step_checks = clean_ohlc(data, reference)
display(pd.Series(accounting, name="rows").to_frame())

print("Rules rechecked after every repair step (raw bounds and adjusted-price consistency):")
display(step_checks)

print("Quarantined rows by reason (unresolved adj_close rows would appear here too):")
display(quarantine["quarantine_reason"].value_counts().rename("rows").to_frame())
display(quarantine[["row_id", "ticker", "ts"] + VALUE_COLUMNS + ["quarantine_reason"]])

,rows
raw_rows,397711
exact_duplicates_removed,29460
conflicting_rows_removed,0
all_prices_missing_removed,0
quarantined,5
retained,368246


Rules rechecked after every repair step (raw bounds and adjusted-price consistency):


,after step,rows,raw prices pending fill,raw bound violations,nonpositive/non-finite raw,adj_close missing/invalid,adj_close inconsistent
0,0. start (normalized),397711,0,5,5,19855,11266
1,1. remove exact duplicates,368251,0,5,5,18374,10418
2,2. resolve conflicting keys,368251,0,5,5,18374,10418
3,3. drop rows missing all prices,368251,0,5,5,18374,10418
4,4. invalid raw prices -> NaN,368251,5,0,0,18374,10418
5,5. forward-fill raw prices,368246,0,0,0,18374,10418
6,6. fill missing volume,368246,0,0,0,18374,10418
7,7. interpolate confirmed spikes,368246,0,0,0,18374,10419
8,8. repair adj_close,368246,0,0,0,0,0


Quarantined rows by reason (unresolved adj_close rows would appear here too):


,rows
quarantine_reason,
leading gap: no earlier price in this ticker to forward-fill,5


,row_id,ticker,ts,open,high,low,close,adj_close,volume,quarantine_reason
0,287343,CVX,1962-01-02,NaN,3.30000,3.24000,3.30000,0.335750,105840,leading gap: no earlier price in this ticker to forward-fill
1,350808,JNJ,1962-01-02,NaN,0.22338,0.22222,0.22338,0.064012,0,leading gap: no earlier price in this ticker to forward-fill
2,224790,MCD,1966-07-05,NaN,0.27366,0.26749,0.26955,0.118090,388800,leading gap: no earlier price in this ticker to forward-fill
3,364568,MRK,1962-01-02,NaN,0.39703,0.39317,0.39537,0.068447,633830,leading gap: no earlier price in this ticker to forward-fill
4,99637,PG,1962-01-02,NaN,1.44000,1.42000,1.43000,0.259300,192000,leading gap: no earlier price in this ticker to forward-fill


Reading the recheck table: the five `open = 0` rows are set to NaN in step 4 (so the bound violations disappear and
they become "pending fill"); step 5 cannot fill them because they are first rows, so they are quarantined.
The corrupted and missing `adj_close` values stay until step 8, which removes all of them. Step 7 shows one
*temporary* extra inconsistency: halving a doubled TRV close changes that day's `adj_close / close` ratio until
step 8 interpolates its `adj_close` from the repaired neighbours. That is exactly why the rules are rechecked after
every step: after the last step every count is zero, and the raw bounds were never broken by a repair.

In [28]:
flag_columns = ["price_filled", "price_interpolated", "volume_filled", "return_outlier", "zero_volume", "weekend_date"]
display(cleaned[flag_columns].sum().rename("rows_flagged").to_frame().T)
display(cleaned["adj_close_repair"].value_counts().rename("rows").to_frame())
print("Interpolated (confirmed error) rows:")
display(cleaned.loc[cleaned["price_interpolated"], KEY + VALUE_COLUMNS + ["repair_note"]])
print("Outlier notes:")
outlier_category = cleaned.loc[cleaned["return_outlier"], "outlier_note"].str.extract(r"kept: ([a-z\- ]+)")[0]
display(outlier_category.str.strip().rename("evidence").value_counts().rename("rows").to_frame())
print("Outliers on days when the adjustment factor moved (corporate action vs vendor blip):")
display(cleaned.loc[cleaned["outlier_note"].str.contains("adjustment factor"), KEY + ["close", "adj_close", "outlier_note"]])

,price_filled,price_interpolated,volume_filled,return_outlier,zero_volume,weekend_date
rows_flagged,0,3,0,5176,74,384


,rows
adj_close_repair,
,339453
was_missing,18373
was_inconsistent,10417
interpolated,3


Interpolated (confirmed error) rows:


,ticker,ts,open,high,low,close,adj_close,volume,repair_note
318327,TRV,1976-10-28,4.650,4.650,4.650,4.650,0.988765,0,confirmed one-day print error (price doubled and reverted); OHLC and adj_close interpolated from neighbouring days
318868,TRV,1978-12-20,4.055,4.055,4.055,4.055,0.915270,0,confirmed one-day print error (price doubled and reverted); OHLC and adj_close interpolated from neighbouring days
319533,TRV,1981-08-10,6.030,6.030,6.030,6.030,1.570000,0,confirmed one-day print error (price doubled and reverted); OHLC and adj_close interpolated from neighbouring days


Outlier notes:


,rows
evidence,
single-stock move,3654
market-wide move,1515
adjustment factor changed,4
one-day blip in the vendor adjustment factor,3


Outliers on days when the adjustment factor moved (corporate action vs vendor blip):


,ticker,ts,close,adj_close,outlier_note
2486,AAPL,1990-10-19,0.28000,0.28000,kept: one-day blip in the vendor adjustment factor (reverts next day); raw close agrees with ohlc
105957,DIS,1974-01-20,0.91502,0.56801,"kept: adjustment factor changed 6% the same day and stays; possible corporate action (split, merger/spin-off, special dividend; not verified)"
105994,DIS,1974-03-05,1.07000,0.66484,"kept: adjustment factor changed 6% the same day and stays; possible corporate action (split, merger/spin-off, special dividend; not verified)"
106253,DIS,1975-03-02,0.87863,0.54542,kept: one-day blip in the vendor adjustment factor (reverts next day); raw close agrees with ohlc
106273,DIS,1975-03-25,0.93581,0.58092,kept: one-day blip in the vendor adjustment factor (reverts next day); raw close agrees with ohlc
155157,IBM,1975-01-28,10.73000,2.65000,"kept: adjustment factor changed 7% the same day and stays; possible corporate action (split, merger/spin-off, special dividend; not verified)"
264444,MSFT,2004-11-15,27.39000,19.05000,"kept: adjustment factor changed 11% the same day and stays; possible corporate action (split, merger/spin-off, special dividend; not verified)"


## Validation

Hard rules for the cleaned table:

* no missing, non-finite or nonpositive values in retained raw and adjusted prices
* unique `(ticker, ts)` keys
* finite, nonnegative volume
* valid price relationships `low <= open, close <= high`
* `adj_close` consistent with its neighbours (no corrupted factors left)
* `ts` stored as ISO `YYYY-MM-DD` text, so SQL date filters work

In [29]:
def validate_cleaned(data: pd.DataFrame) -> pd.Series:
    '''Return a pass/fail Series for every hard rule the cleaned table must satisfy.'''
    prices = data[ALL_PRICE_COLUMNS]
    checks = {
        "prices present, finite and positive": bool(np.isfinite(prices).all().all() and (prices > 0).all().all()),
        "unique (ticker, ts) keys": not data.duplicated(subset=KEY).any(),
        "volume finite and nonnegative": bool(np.isfinite(data["volume"]).all() and (data["volume"] >= 0).all()),
        "low <= open <= high": bool(((data["low"] <= data["open"]) & (data["open"] <= data["high"])).all()),
        "low <= close <= high": bool(((data["low"] <= data["close"]) & (data["close"] <= data["high"])).all()),
        "adj_close factor consistent": not find_inconsistent_adj_close(data).any(),
        "ts is ISO YYYY-MM-DD": bool(data["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
                                     and pd.to_datetime(data["ts"], format="%Y-%m-%d", errors="coerce").notna().all()),
    }
    return pd.Series(checks, name="passed")


validation = validate_cleaned(cleaned)
display(validation.to_frame())
assert validation.all(), "Cleaned data failed validation"

# The repaired adjusted close must now agree with the reference everywhere except the interpolated days
check = cleaned.merge(reference[KEY + ["adj_close"]], on=KEY, suffixes=("", "_ref"))
agrees = np.isclose(check["adj_close"], check["adj_close_ref"], rtol=1e-6)
print("adj_close disagreeing with reference:", (~agrees).sum(),
      "(the interpolated TRV error days, where the reference itself carries the bad print)")

,passed
"prices present, finite and positive",True
"unique (ticker, ts) keys",True
volume finite and nonnegative,True
low <= open <= high,True
low <= close <= high,True
adj_close factor consistent,True
ts is ISO YYYY-MM-DD,True


adj_close disagreeing with reference: 3 (the interpolated TRV error days, where the reference itself carries the bad print)


### Rerun the Data Quality Report after cleaning

The remaining rows are expected and are **not** errors:

* **3σ return outliers, zero volume and weekend dates** are review flags we deliberately keep. There are *more* 3σ flags after
  cleaning, almost all for TRV: removing its three ±100% print errors shrank TRV's standard deviation, so ordinary
  large TRV moves now exceed 3σ. This shows why the screen is a review flag and not a deletion rule.
* **Missing trading days** after cleaning are exactly the quarantined rows: their dates are in the calendar, but we
  chose not to keep a bar we could not repair defensibly.

In [30]:
cleaned_for_report = cleaned.assign(original_ticker=cleaned["ticker"])
quality_report_after = build_quality_report(cleaned_for_report, reference)
issue_summary_after = quality_report_after.groupby(["severity", "issue_type"]).size().rename("issue_rows_after")

comparison = (issue_summary_before.set_index(["severity", "issue_type"])["issue_rows"].rename("issue_rows_before")
              .to_frame().join(issue_summary_after, how="outer").fillna(0).astype(int))
display(comparison)

issue_rows_before  issue_rows_after
severity issue_type                                                        
high     adj_close_inconsistent                     10418                 0
         nonpositive_or_nonfinite_open                  5                 0
         open_below_low                                 5                 0
low      exact_duplicate                            29460                 0
         ticker_markup                              39936                 0
         zero_volume                                   81                74
medium   missing_adj_close                          19855                 0
         missing_trading_day                            0                 5
         return_outlier_3std                         5056              5176
         weekend_date                                 413               384

### Before / after statistics

In [31]:
before_stats = raw[VALUE_COLUMNS].describe().T[["count", "mean", "min", "max"]].add_suffix("_before")
after_stats = cleaned[VALUE_COLUMNS].describe().T[["count", "mean", "min", "max"]].add_suffix("_after")
display(before_stats.join(after_stats).round(3))

rows_per_ticker = (data.groupby("ticker").size().rename("raw_rows").to_frame()
                   .join(cleaned.groupby("ticker").size().rename("cleaned_rows"))
                   .join(reference.groupby("ticker").size().rename("reference_rows")))
display(rows_per_ticker.T)

,count_before,mean_before,min_before,max_before,count_after,mean_after,min_after,max_after
open,397711.0,4.628500e+01,0.000,6.042600e+02,368246.0,4.630900e+01,0.005,6.042600e+02
high,397711.0,4.674800e+01,0.005,6.086300e+02,368246.0,4.677200e+01,0.005,6.086300e+02
low,397711.0,4.581600e+01,0.005,5.978800e+02,368246.0,4.584000e+01,0.005,5.978800e+02
close,397711.0,4.629300e+01,0.005,6.054000e+02,368246.0,4.631700e+01,0.005,6.054000e+02
adj_close,377856.0,5.165200e+01,0.003,1.044251e+04,368246.0,3.727500e+01,0.003,6.054000e+02
volume,397711.0,3.097773e+07,0.000,9.276606e+09,368246.0,3.095184e+07,0.000,9.276606e+09


ticker,AAPL,AMGN,AMZN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,GS,HD,HON,IBM,JNJ,JPM,KO,MCD,MMM,MRK,MSFT,NKE,NVDA,PG,SHW,TRV,UNH,V,VZ,WMT
raw_rows,11942,11290,7458,14260,16980,17044,5525,9427,17073,17151,6931,11731,17111,17138,17097,15145,17147,15830,17041,17111,10486,11989,7030,17108,15470,14325,10909,4516,11151,14295
cleaned_rows,11050,10421,6937,13252,15784,15808,5117,8734,15842,15867,6408,10861,15808,15843,15807,14029,15842,14672,15808,15841,9729,11098,6478,15842,14297,13249,10083,4176,10311,13252
reference_rows,11050,10421,6937,13252,15784,15808,5117,8734,15843,15867,6408,10861,15808,15843,15808,14029,15842,14673,15808,15842,9729,11098,6478,15843,14297,13249,10083,4176,10311,13252


## Save to the database

* `ohlc_hw_cleaned` — cleaned data **with repair flags** (`price_filled`, `price_interpolated`, `volume_filled`,
  `adj_close_repair`, `return_outlier`, `outlier_note`, `zero_volume`, `weekend_date`, `repair_note`) and a unique index on
  `(ticker, ts)`
* `ohlc_hw_quarantine` — rows we could not repair defensibly, with the reason
* `ohlc_hw_ticker_audit` — original ticker label → normalized ticker
* `ohlc_hw_quality_report_before` / `_after` — the diagnosis before repairs and after validation

The raw table `ohlc_hw` is **not** modified.

In [32]:
CLEAN_COLUMNS = (["ts", "ticker"] + VALUE_COLUMNS
                 + ["price_filled", "price_interpolated", "volume_filled", "adj_close_repair",
                    "return_outlier", "outlier_note", "zero_volume", "weekend_date", "repair_note"])
QUARANTINE_COLUMNS = ["row_id", "ts", "ticker", "original_ticker"] + VALUE_COLUMNS + ["quarantine_reason"]

with closing(sqlite3.connect(DB_PATH)) as conn:
    cleaned[CLEAN_COLUMNS].to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_ohlc_hw_cleaned_ticker_ts ON ohlc_hw_cleaned (ticker, ts)")
    quarantine[QUARANTINE_COLUMNS].to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)
    ticker_audit.to_sql("ohlc_hw_ticker_audit", conn, if_exists="replace", index=False)
    quality_report_before.to_sql("ohlc_hw_quality_report_before", conn, if_exists="replace", index=False)
    quality_report_after.to_sql("ohlc_hw_quality_report_after", conn, if_exists="replace", index=False)
    conn.commit()
    raw_after_save = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

# Cell-by-cell comparison with the copy loaded at the start (row_id was added by us, so drop it)
raw_unchanged = raw_after_save.equals(raw.drop(columns="row_id"))
print("Raw table ohlc_hw unchanged (every cell compared):", raw_unchanged)
assert raw_unchanged

Raw table ohlc_hw unchanged (every cell compared): True


In [33]:
# Read the saved table back and validate what is actually stored
with closing(sqlite3.connect(DB_PATH)) as conn:
    stored = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", conn)
    sample_query = pd.read_sql_query(
        "SELECT ts, close, adj_close FROM ohlc_hw_cleaned WHERE ticker = 'AAPL' "
        "AND ts BETWEEN '2024-06-01' AND '2024-06-07' ORDER BY ts", conn)

print(f"Stored rows: {len(stored):,}")
display(validate_cleaned(stored).to_frame())
assert validate_cleaned(stored).all()
display(sample_query)

Stored rows: 368,246


,passed
"prices present, finite and positive",True
"unique (ticker, ts) keys",True
volume finite and nonnegative,True
low <= open <= high,True
low <= close <= high,True
adj_close factor consistent,True
ts is ISO YYYY-MM-DD,True


,ts,close,adj_close
0,2024-06-03,194.03,193.81
1,2024-06-04,194.35,194.13
2,2024-06-05,195.87,195.64
3,2024-06-06,194.48,194.26
4,2024-06-07,196.89,196.66


## Summary of findings

| issue | rows | handling |
|---|---:|---|
| HTML markup in ticker labels | 39,936 | normalized; original labels saved in `ohlc_hw_ticker_audit` |
| exact duplicate records | 29,460 | removed (one copy kept) |
| conflicting `(ticker, ts)` keys | 0 | rule implemented and demonstrated on toy data |
| missing trading days (vs `ohlc` calendar) | 0 | detector demonstrated on toy data |
| missing raw prices / volume | 0 | forward-fill / 0 + flag, demonstrated on toy data |
| missing `adj_close` | 18,374 unique rows (19,855 incl. duplicates) | rebuilt from reference factor of the same date (one more, on a TRV spike day, was interpolated) |
| corrupted `adj_close` (4–24× too large) | 10,417 (+1 TRV spike day flagged by the same detector) | rebuilt from reference factor of the same date |
| `open = 0` on a ticker's first day | 5 | Option 1 → leading gap → **quarantined** |
| one-day doubled prints (TRV 1976, 1978, 1981) | 3 | confirmed errors → interpolated |
| 3σ return outliers | 5,056 before / 5,176 after | **kept** with an evidence note (market-wide / possible corporate action / not verified) |
| zero-volume days | 75 | kept, flagged `zero_volume` (one sits on a quarantined row) |
| Sunday-dated sessions (1974–75; likely Friday sessions stamped on Sunday; reference has the same) | 384 | kept, flagged `weekend_date`; documented uncertainty, not re-dated |

**Row accounting:** 397,711 raw = 368,246 retained + 29,460 exact duplicates removed + 5 quarantined.
All hard validation rules pass on the stored `ohlc_hw_cleaned` table.